# Week 18: Autonomous AI Agents & Multi-Agent Systems
## Notebook 03: Cognitive Agent Memory Systems: Working, Episodic & Semantic Memory

### 1. Conceptual Intuition & Real-World Use Case
Human intelligence relies on multiple memory systems:
1. **Working Memory:** The active scratchpad holding current thoughts (LLM Context Window).
2. **Episodic Memory:** Experiences and past events indexed by time and context.
3. **Semantic Memory:** Structured factual knowledge about the world.

Standard LLMs are stateless: when the context window fills up, older interactions are either abruptly truncated or forgotten. Modern autonomous agents (e.g. MemGPT, Letta) implement a tiered memory hierarchy with **exponential recency decay** and **importance scoring**.

---

### 2. Mathematical Foundations: Memory Retrieval Scoring
Introduced in Generative Agents (Park et al., Stanford/Google), memory retrieval score for memory object $m$ given query $q$ is:
$$\text{Score}(m, q) = \alpha \cdot S_{\text{recency}}(m) + \beta \cdot S_{\text{importance}}(m) + \gamma \cdot S_{\text{relevance}}(m, q)$$

Where:
1. **Recency Decay (Exponential):**
   $$S_{\text{recency}}(m) = e^{-\lambda \cdot \Delta t}$$
   where $\Delta t$ is elapsed hours/days and $\lambda$ is the decay constant.
2. **Importance:** Static score $\in [1, 10]$ assessed by LLM when the memory was created.
3. **Relevance:** Cosine similarity between query embedding and memory embedding.

In [ ]:
from datetime import datetime, timedelta
import numpy as np

# Step 1: Define Memory Class
class MemoryItem:
    def __init__(self, content, importance, timestamp, embedding):
        self.content = content
        self.importance = importance  # 1 to 10 scale
        self.timestamp = timestamp
        self.embedding = embedding / np.linalg.norm(embedding)

# Step 2: Populate Agent Memory Stream
np.random.seed(42)
dim = 4

now = datetime(2026, 10, 25, 12, 0)

memories = [
    MemoryItem(
        content="User mentioned their favorite programming language is Rust.",
        importance=8.0,
        timestamp=now - timedelta(days=5), # 5 days ago
        embedding=np.array([0.9, 0.1, 0.2, 0.0])
    ),
    MemoryItem(
        content="User ordered a pizza with extra cheese.",
        importance=2.0,
        timestamp=now - timedelta(hours=2), # 2 hours ago
        embedding=np.array([0.1, 0.8, 0.1, 0.1])
    ),
    MemoryItem(
        content="User asked for assistance debugging an async Rust network socket.",
        importance=7.0,
        timestamp=now - timedelta(minutes=30), # 30 mins ago
        embedding=np.array([0.85, 0.2, 0.3, 0.1])
    )
]

# Step 3: Implement Tiered Retrieval Function
def retrieve_memories(query_embedding, current_time, alpha=0.3, beta=0.2, gamma=0.5, top_k=2):
    q_norm = query_embedding / np.linalg.norm(query_embedding)
    lambda_decay = 0.05  # decay per day
    
    scored_memories = []
    for m in memories:
        # 1. Recency
        delta_days = (current_time - m.timestamp).total_seconds() / 86400.0
        recency = np.exp(-lambda_decay * delta_days)
        
        # 2. Importance (normalized to 0-1)
        importance_norm = m.importance / 10.0
        
        # 3. Relevance (Cosine Similarity)
        relevance = float(np.dot(q_norm, m.embedding))
        
        # Total Weighted Score
        total = alpha * recency + beta * importance_norm + gamma * relevance
        scored_memories.append((total, recency, importance_norm, relevance, m))
        
    scored_memories.sort(key=lambda x: x[0], reverse=True)
    return scored_memories[:top_k]

# Query: "Which language does the user prefer for systems programming?"
query_vec = np.array([0.92, 0.05, 0.15, 0.0])

results = retrieve_memories(query_vec, now, top_k=2)

print("Top Retrieved Memories for Query:")
for rank, (total, rec, imp, rel, m) in enumerate(results, 1):
    print(f"\nRank {rank} (Total Score: {total:.4f}):")
    print(f"  Content: '{m.content}'")
    print(f"  Breakdown -> Recency: {rec:.3f} | Importance: {imp:.2f} | Relevance: {rel:.3f}")

### 3. Step-by-Step Code Walkthrough
1. **Decay Dynamics:** The pizza memory occurred 2 hours ago (high recency) but had low relevance and low importance, so it was suppressed.
2. **Tri-Factor Balance:** The Rust preference memory occurred 5 days ago, but due to high relevance (0.97) and high importance (0.80), it ranked at the top.
3. **Long-Term Persistence:** This mechanism lets conversational agents recall user preferences across weeks of conversation without blowing up token budgets.

---

### 4. Industry Applications & Trade-offs
- **AI Companions & Executive Assistants:** Remembering family names, personal preferences, and dietary restrictions over months.
- **Trade-off:** Maintaining embedding vector lookups on every single user turn introduces $O(K)$ latency overhead.

---

### 5. Top Interview Questions & Answers
**Q1: How does MemGPT manage memory like an Operating System?**
> **Answer:** MemGPT treats the LLM context window like CPU RAM (Working Memory) and an external vector database like disk storage (Virtual Memory). When working memory reaches capacity, the agent autonomously issues tool calls to page older memories out to disk and retrieve relevant historical facts on demand.

**Q2: Why is raw cosine relevance insufficient for autonomous agent memory?**
> **Answer:** If an agent only uses cosine relevance, recent critical updates can be drowned out by older, highly similar historical records. Balancing recency, importance, and relevance ensures the agent prioritizes both timely updates and core user identity facts.